<a href="https://colab.research.google.com/github/harshitaggarwal01/ssl-time-series/blob/research/masked-ssl/ssl-time-series/notebooks/masked_ssl_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Masked SSL track: training and evaluation walkthrough

Trains the Track 3 candidate, exports it, and scores it on the UCR proxy:

1. Set up the project and the corpus.
2. Build the UCR proxy sets (evaluation only).
3. Train with masked reconstruction plus a crop-view contrastive term.
4. Export the encoder with the untrained features appended.
5. Run the official submission checks.
6. Score on the first and held-out proxy sets.

Runs in Colab (clones the branch) or locally from `notebooks/`. A full run takes roughly 45 to 90 minutes on one GPU; set `SMOKE = True` in step 3 to check the pipeline in a few minutes. Background and results: [Track 3 report](../docs/reports/track3-masked-ssl.md).

## 1. Setup and corpus

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    WORKSPACE = Path("/content/ssl-time-series")
    if not WORKSPACE.exists():
        url = "https://github.com/harshitaggarwal01/ssl-time-series.git"
        subprocess.run(["git", "clone", "--branch", "research/masked-ssl", "--single-branch", url, str(WORKSPACE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    PROJECT = WORKSPACE / "ssl-time-series"
else:
    PROJECT = Path.cwd().parent  # the notebook runs from notebooks/


def run(command):
    """Run a shell command in the project folder and show its output."""
    print("$", command)
    process = subprocess.Popen(
        command, cwd=PROJECT, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"command failed with exit code {process.returncode}")


# The first `uv run` installs the locked environment.
run('uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"')

In [ ]:
# The official corpus: an 8.9 GB download, skipped if corpus.npz is already in data/ or corpus/.
CORPUS = "data" if (PROJECT / "data" / "corpus.npz").exists() else "corpus"
if not (PROJECT / CORPUS / "corpus.npz").exists():
    run("uv run --with gdown gdown 1ld4mifcgwaBZin7cGBNCYm7xXH2T90U6 -O corpus.zip")
    shutil.unpack_archive(PROJECT / "corpus.zip", PROJECT / CORPUS)
    (PROJECT / "corpus.zip").unlink()
print(CORPUS, (PROJECT / CORPUS / "corpus.npz").stat().st_size // 2**20, "MiB")

## 2. UCR proxy sets

The shared synthetic proxy gives 1.000 to every trained model, so models are compared on UCR classification datasets, scored by the official `evaluate.py`. The first set (19 datasets) is for making choices; the held-out set (22 others) is only for confirming them. UCR data is never used for training.

In [ ]:
PROXY_SETS = {"first": ("ucr", ""), "held-out": ("ucr_holdout", "--held-out")}
for folder, flag in PROXY_SETS.values():
    if not (PROJECT / "eval_data" / folder / "tasks.json").exists():
        run(f"uv run python -m scripts.proxy_ucr {flag} --out eval_data/{folder}")

## 3. Train

40% of each window is hidden in spans and reconstructed (smooth L1 on the normalized level); a contrastive term pulls together two random crops of the same window. `--real-only` normalizes and pools over the series rather than its padding. `ARCH = "conv"` is the recommended dilated convolutional encoder; `"transformer"` is the patch transformer of the shared branch.

In [ ]:
SMOKE = False  # True: 20 steps, to check the pipeline end to end
ARCH = "conv"  # or "transformer"

name = f"walkthrough-{ARCH}" + ("-smoke" if SMOKE else "")
arch_flags = "--arch conv --depth 8" if ARCH == "conv" else ""
run(
    f"uv run python -m scripts.train_mae --corpus {CORPUS} --output outputs/{name}.pt {arch_flags} "
    f"--real-only --contrast 1.0 --view crop --steps {20 if SMOKE else 30000} --workers {2 if IN_COLAB else 6}"
)

## 4. Export and validate

The decoder and heads stay behind. `--pooling mean_std` reads out the mean and spread of the encoder's output (640 dimensions) and `--fixed` appends 1,285 untrained features (spectrum bands and random dilated kernels), for 1,925 dimensions in total.

In [ ]:
run(f"uv run python -m scripts.export_model --checkpoint outputs/{name}.pt --output outputs/{name}.onnx --pooling mean_std --fixed")
run(f"uv run python test_submission.py outputs/{name}.onnx")

## 5. Evaluate

Mean macro one-vs-rest ROC AUC over the tasks of each set. Reference values from the report (30,000 steps, mean of seeds 0 and 1):

| | First | Held-out |
| --- | --- | --- |
| Convolutional candidate | 0.933 | 0.921 |
| Transformer candidate | 0.936 | 0.918 |
| Untrained features only, no encoder | 0.928 | 0.915 |

A smoke run lands near the last row: most of the score comes from the untrained features.

In [ ]:
for label, (folder, _) in PROXY_SETS.items():
    scores = PROJECT / "outputs" / f"{name}.{folder}.json"
    run(f"uv run python evaluate.py outputs/{name}.onnx eval_data/{folder} --out outputs/{scores.name}")
    aucs = [task["auc"] for task in json.loads(scores.read_text())["tasks"].values()]
    print(f"{label}: mean AUC {sum(aucs) / len(aucs):.4f} over {len(aucs)} tasks\n")